# Weighing Sgr A*: the orbit of the star S2

*astro · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/astro/sgr_a_star_orbits)

The star S2 circles the centre of the Milky Way every 16 years, passing within 120 AU of an invisible mass. A Keplerian orbit, with Kepler's equation solved by vectorised Newton iterations, is fitted to 25 years of real positions and radial velocities. Levenberg–Marquardt with autograd Jacobians runs from 64 random starts at once and recovers a black hole of about $4.3\times 10^6\,M_\odot$ at $R_0 \approx 8.3$ kpc. The fit then predicts the two relativistic effects GRAVITY later detected: about 200 km/s of redshift at pericentre and $12'$ of precession per orbit.

**Vector form**

$$E - e\sin E = \frac{2\pi}{P}(t - t_p),\qquad \mathbf r = R_z(\Omega)R_x(i)R_z(\omega)\begin{pmatrix} a(\cos E - e)\\ a\sqrt{1-e^2}\sin E\\ 0\end{pmatrix},\qquad P^2 = \frac{4\pi^2 (aR_0)^3}{GM}$$

**Index (tensor) form**

$$E_{bt} \leftarrow E_{bt} - \frac{E_{bt} - e_b\sin E_{bt} - M_{bt}}{1 - e_b\cos E_{bt}},\qquad r_{bti} = R_{bij}\,q_{btj},\qquad \chi^2_b = \sum_n \Big(\frac{y_{bn} - y_n^{\mathrm{obs}}}{\sigma_n}\Big)^2,\qquad (J^\top J + \lambda\,\mathrm{diag})_{bkl}\,\delta_{bl} = -J_{bnk}\,r_{bn}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $p_b$ | `random_starts(B, gen)` | (B, 8) | one orbit per start: $a$, $e$, $i$, $\Omega$, $\omega$, $t_p$, $M$, $R_0$ |
| $E_{bt}$ | `kepler_solve(n * (t - tp), e)` | (B, T) | eccentric anomaly by Newton iterations, every orbit and epoch at once |
| $R_{bij}$ | `rotation(p)` | (B, 3, 3) | orbital plane → (north, east, line of sight), built with einsum |
| $r_{bti}$ | `einsum('bij,btj->bti', R, q)` | (B, T, 3) | $j$ summed: position of S2 in AU; divided by $R_0$ gives mas on the sky |
| $v_{bt3}$ | `observables(p, t)[2]` | (B, T) | line-of-sight velocity in km/s: this is what fixes $R_0$ |
| $r_{bn}$ | `residuals(theta, data)` | (B, 334) | (model − data)/$\sigma$ for 145 RA, 145 Dec and 44 velocities |
| $J_{bnk}$ | `jacobian(theta, data)` | (B, 334, 8) | forward-mode autograd, vmapped over the 8 parameter directions |
| $\delta_{bl}$ | `torch.linalg.solve(JtJ + damp, -grad)` | (B, 8) | one Levenberg–Marquardt step for every start at once |
| $\Delta\varphi$ | `precession_per_orbit(M, a, e)` | () | $6\pi GM/(c^2a(1-e^2))$: Schwarzschild precession, $\approx 12'$ for S2 |

Since the early 1990s two teams, led by Reinhard Genzel (MPE, ESO telescopes) and Andrea Ghez (UCLA, Keck), have followed stars in the central arcsecond of the Galaxy. S2 is the brightest star with a short orbit. Its 16-year ellipse fixes the enclosed mass in a volume too small for anything but a black hole. Positions give the orbit's angular size and period, and the radial velocities (Doppler shifts of spectral lines) give its true size in km. Together they yield the distance $R_0$ and, through Kepler's third law, the mass. The model has 8 parameters: six elements plus $M$ and $R_0$ (the period follows from $P^2 = 4\pi^2(aR_0)^3/GM$). The fit uses Levenberg–Marquardt with Jacobians from autograd, run from many random starting orbits at once. Many of them fall into local minima, which is why a batch of starts is needed, and the best $\chi^2$ is kept. Kepler's equation is solved by Newton iterations on detached tensors plus one final live step, so autograd sees the exact implicit derivative. Result with the defaults: $M \approx 4.32 \pm 0.15 \times 10^6\,M_\odot$, $R_0 \approx 8.35 \pm 0.12$ kpc, $P \approx 16.03$ yr, $e \approx 0.884$, with $\chi^2 \approx 336$ for 334 data. Published values: $4.28 \pm 0.10 \pm 0.21 \times 10^6\,M_\odot$ and $8.32 \pm 0.07 \pm 0.14$ kpc from 17 stars (Gillessen et al. 2017). The fit then predicts two relativistic effects that it does not include. At pericentre (120 AU, 7700 km/s, 2.6 % of $c$) the gravitational redshift plus the transverse Doppler shift add about 200 km/s; GRAVITY measured $f = 0.90 \pm 0.09\,(\mathrm{stat}) \pm 0.15\,(\mathrm{sys})$ of that GR value after the May 2018 pericentre (GRAVITY Collaboration 2018). The orbit also precesses by $6\pi GM/(c^2a(1-e^2)) \approx 12'$ per revolution, measured as $f_{SP} = 1.10 \pm 0.19$ of the GR value (GRAVITY Collaboration 2020). These are cited detections, not computed from these data: the 2016 table ends before the 2018 pericentre. Simplifications: a pure Kepler orbit, with no Rømer delay (light-travel time across the orbit, about 8 days), no relativistic terms and no drift of the reference frame (the paper fits 5 extra frame parameters). Data: Gillessen et al. 2017, ApJ 837, 30 (arXiv:1611.09144), table 5 of S2 data from VizieR/CDS catalogue J/ApJ/837/30 (doi:10.26093/cds/vizier.18370030), used with citation under the CDS conditions of use. Prizes: Shaw Prize in Astronomy 2008 (Genzel), Crafoord Prize in Astronomy 2012 (Genzel and Ghez) and Nobel Prize in Physics 2020 (half to Genzel and Ghez, for the discovery of a supermassive compact object at the centre of our galaxy).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/astro/sgr_a_star_orbits/main.py
"""Main loop: S2 positions + radial velocities → batched Levenberg–Marquardt from many starts → M, R0 → GR effects."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli

from . import plot, solve, tensors
from .equation import (
    AU,
    GM_SUN,
    C,
    kepler_solve,
    pericentre_redshift_kms,
    period,
    precession_per_orbit,
    semi_major_axis_au,
    state_vectors,
)


def run(params: dict, out_dir: Path) -> Path:
    # CPU, float64: e = 0.88 makes Kepler's equation stiff near pericentre, and χ² values reach 10⁷ at bad starts.
    data = tensors.load_s2()
    print(
        f"S2: {len(data['t_a'])} positions ({data['t_a'][0]:.1f}–{data['t_a'][-1]:.1f}) and "
        f"{len(data['t_v'])} radial velocities ({data['t_v'][0]:.1f}–{data['t_v'][-1]:.1f}), Gillessen et al. 2017"
    )
    e = torch.linspace(0.0, 0.99, 50, dtype=torch.float64)[:, None]
    mean = torch.linspace(-math.pi, math.pi, 1001, dtype=torch.float64)
    big_e = kepler_solve(mean, e)
    print(
        f"Kepler solver: max |E − e sin E − M| = {(big_e - e * torch.sin(big_e) - mean).abs().max():.1e} "
        "on 50 × 1001 cases"
    )

    gen = torch.Generator().manual_seed(params["seed"])
    fit = solve.fit(data, n_starts=params["n_starts"], n_iter=params["n_iter"], gen=gen)
    best, err, chi2 = fit["best"], fit["errors"], fit["chi2"]
    good = int((chi2 < chi2.min() + 10).sum())
    print(
        f"\n{params['n_starts']} starts fitted at once; {good} reach the best χ² (within 10): "
        f"χ²_min = {chi2.min():.1f} for {data['n_res']} data"
    )
    labels = {"a": "a (mas)", "e": "e", "inc": "i (°)", "Omega": "Ω (°)", "omega": "ω (°)", "tp": "t_p (yr)",
              "P": "P (yr)", "M": "M (10⁶ M☉)", "R0": "R0 (kpc)"}  # fmt: skip
    for key, label in labels.items():
        print(f"  {label:12s} {best[key]:10.4f} ± {err[key]:.4f}")
    print(
        "published (17 stars, Gillessen et al. 2017): M = 4.28 ± 0.10 (stat) ± 0.21 (sys) × 10⁶ M☉, "
        "R0 = 8.32 ± 0.07 ± 0.14 kpc; S2 period 16.00 yr"
    )

    p = fit["p"][fit["k"] : fit["k"] + 1]
    a_au = semi_major_axis_au(p).item()
    r_p = a_au * (1 - best["e"])
    r_s = 2 * GM_SUN * best["M"] * 1e6 / C**2 / AU  # Schwarzschild radius in AU
    t = torch.linspace(best["tp"] - 0.05, best["tp"] + 0.05, 2001, dtype=torch.float64)
    _, v = state_vectors(p, t)
    v_max = v.norm(dim=-1).max().item() * AU / (365.25 * 86400) / 1000
    print(
        f"\npericentre {r_p:.0f} AU = {r_p * AU / C / 3600:.1f} light-hours = {r_p / r_s:.0f} Schwarzschild radii, "
        f"speed {v_max:.0f} km/s ({v_max / 2997.92:.2f} % of c)"
    )
    print(f"Kepler's third law: P = 2π√(a³/GM) = {period(p).item():.3f} yr with a = {a_au:.0f} AU")
    grav, doppler = pericentre_redshift_kms(p)
    print(
        f"GR prediction at pericentre: gravitational redshift {grav.item():.0f} km/s + transverse Doppler "
        f"{doppler.item():.0f} km/s = {grav.item() + doppler.item():.0f} km/s; GRAVITY 2018 measured "
        f"f = 0.90 ± 0.09 (stat) ± 0.15 (sys) of the GR value (z ≈ 200 km/s / c)"
    )
    dphi = precession_per_orbit(torch.tensor(best["M"] * 1e6), torch.tensor(a_au), torch.tensor(best["e"]))
    print(
        f"Schwarzschild precession 6πGM/(c²a(1−e²)) = {math.degrees(dphi.item()) * 60:.1f}′ per orbit; "
        f"GRAVITY 2020 measured f_SP = 1.10 ± 0.19 of the GR value (≈ 12′)"
    )
    return plot.animate(data, fit, out_dir / "sgr_a_star_orbits.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/astro/sgr_a_star_orbits/equation.py
r"""Physics → tensor translation for a Keplerian orbit seen on the sky.

Units: time in years, lengths in AU, angles on the sky in milliarcseconds (mas), masses in M☉, distance R0 in kpc.
An angle of a mas at R0 kpc is a·R0 AU (1″ at 1 pc is 1 AU), so the semi-major axis is a_AU = a_mas · R0_kpc and
Kepler's third law reads P = 2π √(a³/GM), with GM ≈ 4π² AU³/yr² per solar mass.

Orbit in its own plane (eccentric anomaly E from Kepler's equation M = E − e sin E, mean anomaly M = 2π(t − t_p)/P):

    X = a (cos E − e),  Y = a √(1 − e²) sin E,   Ẋ = −a n sin E/(1 − e cos E),  Ẏ = a n √(1 − e²) cos E/(1 − e cos E)

Rotated onto the sky by R = R_z(Ω) R_x(i) R_z(ω):   r_{bti} = R_{bij} q_{btj}   (j summed), with i = (north = ΔDec,
east = ΔRA, line of sight away from us; the usual visual-binary convention). RV = v_z, positive = receding.
"""

import math

import torch

GM_SUN = 1.32712440018e20  # m³/s²
AU = 1.495978707e11  # m
YEAR = 365.25 * 86400.0  # Julian year, s
C = 299_792_458.0  # m/s
AU_PER_YR_KMS = AU / YEAR / 1000.0  # 1 AU/yr = 4.74 km/s
RV_SIGN = 1.0  # orbit-astronomy convention: the third axis points away from us, so RV = +v_z


def gm_au3_yr2(mass_msun: torch.Tensor) -> torch.Tensor:
    """GM in AU³/yr² for a mass in M☉ (≈ 4π² per M☉)."""
    return GM_SUN * mass_msun * YEAR**2 / AU**3


def semi_major_axis_au(p: torch.Tensor) -> torch.Tensor:
    """a_AU = a_mas · R0_kpc.   p (B, 8) = [a_mas, e, i°, Ω°, ω°, t_p, M/10⁶ M☉, R0_kpc] → (B,)"""
    return p[:, 0] * p[:, 7]


def period(p: torch.Tensor) -> torch.Tensor:
    """Kepler's third law P = 2π √(a³/GM) in years: (B,)."""
    return 2 * math.pi * torch.sqrt(semi_major_axis_au(p) ** 3 / gm_au3_yr2(p[:, 6] * 1e6))


def kepler_solve(mean: torch.Tensor, e: torch.Tensor, n_iter: int = 30) -> torch.Tensor:
    """E with E − e sin E = M, for any broadcastable shapes, by vectorised Newton iterations.

    The iterations run on detached tensors; one final Newton step on the live tensors then carries the exact
    implicit-function derivatives (dE/dM = 1/(1 − e cos E), dE/de = sin E/(1 − e cos E)) for autograd.
    """
    m0, e0 = mean.detach(), e.detach()
    big_e = m0 + 0.85 * e0 * torch.sign(torch.sin(m0))  # a start that converges for every e < 1
    for _ in range(n_iter):
        big_e = big_e - (big_e - e0 * torch.sin(big_e) - m0) / (1 - e0 * torch.cos(big_e))
    big_e = big_e.detach()
    return big_e - (big_e - e * torch.sin(big_e) - mean) / (1 - e * torch.cos(big_e))


def rotation(p: torch.Tensor) -> torch.Tensor:
    """R = R_z(Ω) R_x(i) R_z(ω) from the orbital plane to (north, east, away from us): (B, 3, 3)."""
    inc, big_om, om = (torch.deg2rad(p[:, k]) for k in (2, 3, 4))
    zero, one = torch.zeros_like(inc), torch.ones_like(inc)

    def rz(a):
        return torch.stack([torch.cos(a), -torch.sin(a), zero, torch.sin(a), torch.cos(a), zero, zero, zero, one], -1)

    rx = torch.stack([one, zero, zero, zero, torch.cos(inc), -torch.sin(inc), zero, torch.sin(inc), torch.cos(inc)], -1)
    shape = (-1, 3, 3)
    return torch.einsum("bij,bjk,bkl->bil", rz(big_om).view(shape), rx.view(shape), rz(om).view(shape))


def state_vectors(p: torch.Tensor, t: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Position (AU) and velocity (AU/yr) in (north, east, away from us).   p (B, 8), t (T,) → (B, T, 3) each"""
    a, e, tp = semi_major_axis_au(p)[:, None], p[:, 1:2], p[:, 5:6]
    n = 2 * math.pi / period(p)[:, None]  # mean motion, rad/yr
    big_e = kepler_solve(n * (t - tp), e)  # (B, T)
    cos_e, sin_e, root = torch.cos(big_e), torch.sin(big_e), torch.sqrt(1 - e**2)
    denom = 1 - e * cos_e
    zeros = torch.zeros_like(big_e)
    q = torch.stack([a * (cos_e - e), a * root * sin_e, zeros], -1)  # (B, T, 3) in the orbital plane
    dq = torch.stack([-a * n * sin_e / denom, a * n * root * cos_e / denom, zeros], -1)
    rot = rotation(p)
    return torch.einsum("bij,btj->bti", rot, q), torch.einsum("bij,btj->bti", rot, dq)  # j summed


def observables(p: torch.Tensor, t: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """What the telescopes measure: ΔRA (mas), ΔDec (mas), radial velocity (km/s), each (B, T)."""
    r, v = state_vectors(p, t)
    to_mas = 1.0 / p[:, 7:8]  # AU → mas at R0 kpc
    return r[..., 1] * to_mas, r[..., 0] * to_mas, RV_SIGN * v[..., 2] * AU_PER_YR_KMS


def precession_per_orbit(mass_msun: torch.Tensor, a_au: torch.Tensor, e: torch.Tensor) -> torch.Tensor:
    r"""Schwarzschild periapsis advance Δφ = 6πGM / (c² a (1 − e²)) in radians per orbit."""
    return 6 * math.pi * GM_SUN * mass_msun / (C**2 * a_au * AU * (1 - e**2))


def pericentre_redshift_kms(p: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    r"""Gravitational redshift GM/(c r_p) and transverse Doppler v_p²/(2c) at pericentre, as velocities (km/s)."""
    a = semi_major_axis_au(p) * AU
    e = p[:, 1]
    gm = GM_SUN * p[:, 6] * 1e6
    r_p = a * (1 - e)
    v_p2 = gm / a * (1 + e) / (1 - e)  # vis-viva at pericentre
    return gm / (r_p * C) / 1000, v_p2 / (2 * C) / 1000


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/astro/sgr_a_star_orbits/tensors.py
"""Build the concrete tensors: the S2 data table, the parameter packing, random starting orbits and synthetic data.

data/s2.csv holds the S2 measurements used by Gillessen et al. 2017 (ApJ 837, 30), table 5 from VizieR/CDS
(J/ApJ/837/30): 145 astrometric epochs (offsets from Sgr A* in RA and Dec, mas, 1992–2016, NTT and VLT) and 44
radial velocities (km/s, 2000–2016, VLT and Keck). Everything is float64.

Parameter vector p (B, 8) = [a (mas), e, i (°), Ω (°), ω (°), t_p (yr), M (10⁶ M☉), R0 (kpc)]. The fit works on an
unconstrained copy θ = [ln a, logit e, i, Ω, ω, t_p, ln M, ln R0] (angles in degrees), so a > 0, 0 < e < 1, M, R0 > 0.
"""

from pathlib import Path

import torch

DATA = Path(__file__).parent / "data" / "s2.csv"
DTYPE = torch.float64
NAMES = ("a", "e", "inc", "Omega", "omega", "tp", "M", "R0")


def load_s2(path: Path = DATA) -> dict:
    """Astrometry {t_a, ra, dec, ra_err, dec_err} (Na,) and radial velocities {t_v, rv, rv_err} (Nv,)."""
    lines = [ln for ln in Path(path).read_text(encoding="utf-8").splitlines() if ln.strip() and ln[0] != "#"]
    header = lines[0].split(",")
    rows = [dict(zip(header, ln.split(","))) for ln in lines[1:]]
    astro = [r for r in rows if r["kind"] == "a"]
    vel = [r for r in rows if r["kind"] == "rv"]

    def col(rs, key):
        return torch.tensor([float(r[key]) for r in rs], dtype=DTYPE)

    data = {
        "t_a": col(astro, "epoch"), "ra": col(astro, "ra_mas"), "ra_err": col(astro, "ra_err"),
        "dec": col(astro, "dec_mas"), "dec_err": col(astro, "dec_err"),
        "t_v": col(vel, "epoch"), "rv": col(vel, "rv_kms"), "rv_err": col(vel, "rv_err"),
    }  # fmt: skip
    data["n_res"] = 2 * len(astro) + len(vel)
    return data


def pack(values: dict) -> torch.Tensor:
    """Physical parameters from a dict with keys NAMES: (8,)."""
    return torch.tensor([float(values[k]) for k in NAMES], dtype=DTYPE)


def unconstrained(p: torch.Tensor) -> torch.Tensor:
    """θ from p: (B, 8) → (B, 8)."""
    e = p[:, 1]
    return torch.stack([p[:, 0].log(), torch.log(e / (1 - e)), p[:, 2], p[:, 3], p[:, 4], p[:, 5],
                        p[:, 6].log(), p[:, 7].log()], -1)  # fmt: skip


def constrained(theta: torch.Tensor) -> torch.Tensor:
    """p from θ: (B, 8) → (B, 8)."""
    return torch.stack([theta[:, 0].exp(), torch.sigmoid(theta[:, 1]), theta[:, 2], theta[:, 3], theta[:, 4],
                        theta[:, 5], theta[:, 6].exp(), theta[:, 7].exp()], -1)  # fmt: skip


def random_starts(n: int, gen: torch.Generator) -> torch.Tensor:
    """n broad random starts: a 60–250 mas, e 0.3–0.95, any orientation, t_p 1995–2012, M 2–8, R0 6–11 kpc: (n, 8)."""
    u = torch.rand(n, 8, generator=gen, dtype=DTYPE)
    lo = torch.tensor([60.0, 0.3, 0.0, 0.0, 0.0, 1995.0, 2.0, 6.0], dtype=DTYPE)
    hi = torch.tensor([250.0, 0.95, 180.0, 360.0, 360.0, 2012.0, 8.0, 11.0], dtype=DTYPE)
    return lo + (hi - lo) * u


def synthetic_s2(data: dict, p: torch.Tensor, gen: torch.Generator) -> dict:
    """The same epochs and error bars, with values from the orbit p (8,) plus Gaussian noise of those sizes."""
    from .equation import observables  # local import keeps this file about tensors

    ra, dec, _ = observables(p[None], data["t_a"])
    _, _, rv = observables(p[None], data["t_v"])
    out = dict(data)

    def noisy(x, err):
        return x[0] + err * torch.randn(err.shape, generator=gen, dtype=DTYPE)

    out["ra"], out["dec"] = noisy(ra, data["ra_err"]), noisy(dec, data["dec_err"])
    out["rv"] = noisy(rv, data["rv_err"])
    return out


def wrap_degrees(x: torch.Tensor) -> torch.Tensor:
    """Angles into [0, 360)."""
    return torch.remainder(x, 360.0)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/astro/sgr_a_star_orbits/solve.py
"""Fit the orbit: weighted residuals, their Jacobian by autograd, and batched Levenberg–Marquardt from many starts."""

import torch
from torch.func import jvp, vmap

from problems._common.progress import steps

from . import tensors
from .equation import observables, period


def residuals(theta: torch.Tensor, data: dict) -> torch.Tensor:
    """(model − data)/σ for every RA, Dec and RV measurement: θ (B, 8) → (B, R), R = 2 Na + Nv."""
    p = tensors.constrained(theta)
    ra, dec, _ = observables(p, data["t_a"])
    _, _, rv = observables(p, data["t_v"])
    return torch.cat(
        [(ra - data["ra"]) / data["ra_err"], (dec - data["dec"]) / data["dec_err"], (rv - data["rv"]) / data["rv_err"]],
        dim=-1,
    )


def jacobian(theta: torch.Tensor, data: dict) -> torch.Tensor:
    """J_{brk} = ∂r_{br}/∂θ_{bk} by forward-mode autograd: (B, R, 8).

    One Jacobian-vector product per parameter direction k (θ_k nudged in every orbit of the batch at once; the
    orbits are independent), and the 8 directions themselves are vectorised with vmap.
    """
    directions = torch.eye(theta.shape[-1], dtype=theta.dtype)[:, None, :].expand(-1, *theta.shape)  # (8, B, 8)

    def column(v):  # ∂r/∂θ · v for the whole batch: (B, R)
        return jvp(lambda th: residuals(th, data), (theta,), (v,))[1]

    return vmap(column)(directions).permute(1, 2, 0)


def fit(data: dict, n_starts: int = 256, n_iter: int = 60, gen: torch.Generator | None = None) -> dict:
    """Levenberg–Marquardt on every starting orbit at once; returns all fits and the best one.

    Each step solves (JᵀJ + λ diag JᵀJ) δ = −Jᵀr for the whole batch (a (B, 8, 8) solve); λ shrinks after an
    accepted step and grows after a rejected one, separately for every start.
    """
    gen = gen or torch.Generator().manual_seed(0)
    theta = tensors.unconstrained(tensors.random_starts(n_starts, gen))  # (B, 8)
    lam = torch.full((n_starts,), 1e-2, dtype=theta.dtype)
    chi2 = residuals(theta, data).pow(2).sum(-1)
    chi2 = torch.nan_to_num(chi2, nan=torch.inf)
    history = [chi2.clone()]
    for _ in steps(n_iter, "Levenberg–Marquardt"):
        r = residuals(theta, data)  # (B, R)
        jac = torch.nan_to_num(jacobian(theta, data))  # (B, R, 8)
        jtj = torch.einsum("brk,brl->bkl", jac, jac)  # r summed
        grad = torch.einsum("brk,br->bk", jac, torch.nan_to_num(r))
        damp = lam[:, None, None] * torch.diag_embed(torch.diagonal(jtj, dim1=-2, dim2=-1) + 1e-9)
        delta = torch.linalg.solve(jtj + damp, -grad[..., None])[..., 0]
        trial = theta + delta
        chi2_trial = torch.nan_to_num(residuals(trial, data).pow(2).sum(-1), nan=torch.inf)
        better = chi2_trial < chi2
        theta = torch.where(better[:, None], trial, theta)
        chi2 = torch.where(better, chi2_trial, chi2)
        lam = torch.where(better, lam / 3, lam * 4).clamp(1e-9, 1e9)
        history.append(chi2.clone())
    p = tensors.constrained(theta)
    k = int(chi2.argmin())
    best = {name: p[k, i].item() for i, name in enumerate(tensors.NAMES)}
    if best["inc"] % 360.0 > 180.0:  # (i, Ω, ω) and (360° − i, Ω + 180°, ω + 180°) are the same orbit
        best["inc"], best["Omega"], best["omega"] = (
            360.0 - best["inc"] % 360.0,
            best["Omega"] + 180,
            best["omega"] + 180,
        )
    for key in ("inc", "Omega", "omega"):
        best[key] = best[key] % 360.0
    best["P"] = period(p[k : k + 1]).item()
    return {"p": p, "chi2": chi2, "history": torch.stack(history), "best": best, "k": k,
            "errors": errors(theta[k : k + 1], data, chi2[k].item())}  # fmt: skip


def errors(theta: torch.Tensor, data: dict, chi2: float) -> dict:
    """1σ errors of the physical parameters from the covariance (JᵀJ)⁻¹, scaled by χ²/dof, via the chain rule."""
    jac = jacobian(theta, data)[0]  # (R, 8)
    cov_theta = torch.linalg.inv(jac.T @ jac) * max(1.0, chi2 / (data["n_res"] - 8))
    dp = torch.func.jacfwd(lambda th: tensors.constrained(th[None])[0])(theta[0])  # (8, 8) ∂p/∂θ
    cov = dp @ cov_theta @ dp.T
    sig = cov.diagonal().sqrt()
    out = {name: sig[i].item() for i, name in enumerate(tensors.NAMES)}
    dper = torch.func.jacfwd(lambda th: period(tensors.constrained(th[None]))[0])(theta[0])
    out["P"] = (dper @ cov_theta @ dper).sqrt().item()
    return out


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/astro/sgr_a_star_orbits/plot.py
"""Animate S2 around Sgr A*: the ellipse on the sky with the data, the radial-velocity curve, the fit convergence."""

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from .equation import observables, period

N_FRAMES = 72


def animate(data: dict, fit: dict, path: Path) -> Path:
    k = fit["k"]
    p = fit["p"][k : k + 1]
    best = fit["best"]
    big_p = period(p).item()
    # times of one orbit, evenly spaced in eccentric anomaly so the fast pericentre passage gets frames too
    big_e = torch.linspace(-math.pi, math.pi, N_FRAMES, dtype=torch.float64)
    t_frames = best["tp"] + (big_e - best["e"] * torch.sin(big_e)) * big_p / (2 * math.pi)
    t_dense = torch.linspace(1990.0, 2020.0, 3000, dtype=torch.float64)
    ra_c, dec_c, rv_c = (x[0].numpy() for x in observables(p, t_dense))
    ra_f, dec_f, rv_f = (x[0].numpy() for x in observables(p, t_frames))
    tf, td = t_frames.numpy(), t_dense.numpy()

    fig, (ax_s, ax_v, ax_c) = plt.subplots(1, 3, figsize=(12, 4.3), gridspec_kw={"width_ratios": [1.05, 1.25, 0.8]})
    ax_s.errorbar(data["ra"].numpy(), data["dec"].numpy(), xerr=data["ra_err"].numpy(), yerr=data["dec_err"].numpy(),
                  fmt="o", ms=2.5, color="#93c5fd", ecolor="#bfdbfe", elinewidth=0.8, zorder=1,
                  label="S2 astrometry 1992–2016")  # fmt: skip
    ax_s.plot(ra_c, dec_c, color="#2563eb", lw=1.4, zorder=2, label="Kepler fit")
    ax_s.plot([0], [0], marker="+", ms=12, mew=2.5, color="#111827", zorder=4)
    ax_s.text(-6, 6, "Sgr A*", fontsize=8)
    (trail,) = ax_s.plot([], [], color="#dc2626", lw=2.2, zorder=3)
    (star,) = ax_s.plot([], [], "o", ms=8, color="#dc2626", mec="#7f1d1d", zorder=5)
    ax_s.set_xlim(ra_c.max() + 25, ra_c.min() - 25)  # east to the left, as on the sky
    ax_s.set_ylim(dec_c.min() - 20, dec_c.max() + 20)
    ax_s.set_aspect("equal")
    ax_s.set_xlabel("ΔRA (mas, east ←)")
    ax_s.set_ylabel("ΔDec (mas)")
    ax_s.legend(loc="center", fontsize=7)
    title_s = ax_s.set_title("", fontsize=9)

    ax_v.errorbar(data["t_v"].numpy(), data["rv"].numpy(), yerr=data["rv_err"].numpy(), fmt="o", ms=3,
                  color="#2563eb", ecolor="#93c5fd", label="radial velocity (VLT, Keck)")  # fmt: skip
    ax_v.plot(td, rv_c, color="#111827", lw=1.2, label="Kepler fit")
    ax_v.axvline(best["tp"], color="#9ca3af", ls="--", lw=1)
    ax_v.axvline(best["tp"] + big_p, color="#9ca3af", ls="--", lw=1)
    ax_v.text(
        best["tp"] + big_p - 0.3, rv_c.min(), "next pericentre\n(GRAVITY 2018)", fontsize=7, ha="right", color="#4b5563"
    )
    (marker,) = ax_v.plot([], [], "o", ms=8, color="#dc2626", mec="#7f1d1d", zorder=5)
    ax_v.set_xlim(1999, 2019.5)
    ax_v.set_ylim(rv_c.min() - 300, rv_c.max() + 300)
    ax_v.set_xlabel("year")
    ax_v.set_ylabel("line-of-sight velocity (km/s)")
    ax_v.legend(loc="upper center", fontsize=7)
    ax_v.set_title(
        f"M = {best['M']:.2f} ± {fit['errors']['M']:.2f} × 10⁶ M☉,  "
        f"R0 = {best['R0']:.2f} ± {fit['errors']['R0']:.2f} kpc,"
        f"  P = {big_p:.2f} yr",
        fontsize=9,
    )

    hist = fit["history"].numpy()  # (iterations + 1, B)
    n_res = data["n_res"]
    for b in range(hist.shape[1]):
        ax_c.plot(hist[:, b] / n_res, color="#d1d5db", lw=0.7)
    ax_c.plot(hist[:, k] / n_res, color="#dc2626", lw=2, label="best start")
    ax_c.axhline(1.0, color="#16a34a", ls="--", lw=1, label="χ²/N = 1")
    ax_c.set_yscale("log")
    ax_c.set_xlabel("Levenberg–Marquardt step")
    ax_c.set_ylabel("χ² / N")
    ax_c.legend(loc="upper right", fontsize=7)
    ax_c.set_title(f"{hist.shape[1]} starts fitted in one batch", fontsize=9)
    for ax in (ax_s, ax_v, ax_c):
        ax.tick_params(labelsize=7)

    def update(i: int):
        lo = max(0, i - 12)
        trail.set_data(ra_f[lo : i + 1], dec_f[lo : i + 1])
        star.set_data([ra_f[i]], [dec_f[i]])
        t_now = tf[i] if tf[i] > 1999 else tf[i] + big_p  # show the RV marker inside the data window
        rv_now = np.interp(t_now, td, rv_c)
        marker.set_data([t_now], [rv_now])
        title_s.set_text(f"S2 around Sgr A*, year {tf[i]:.2f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(N_FRAMES, N_FRAMES), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_starts': 64,
    'n_iter': 40,
    'seed': 0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.astro.sgr_a_star_orbits")]:
    del sys.modules[name]
main = importlib.import_module("problems.astro.sgr_a_star_orbits.main")
problem_dir = Path("problems/astro/sgr_a_star_orbits")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
